# HF2LI Total Harmonic Distortion (THD) Measurement

Sweeps the drive frequency and simultaneously measures the **1st, 2nd, 3rd and 4th harmonics**
using four demodulators locked to the same oscillator. THD is calculated at each frequency point.

**Hardware connection:**
```
Signal Output 1  →  Device Under Test (DUT)  →  Signal Input 1
```
For a self-test (loopback), connect Output 1 directly to Input 1.

**Requirements:** HF2LI DEV182253 · USB · LabOne running locally · `pip install zhinst-core`

## 1. Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zhinst.core as zc

## 2. Parameters

Adjust these settings to match your setup before running.

In [ ]:
DEVICE_ID = "dev182253"          # HF2LI device ID — use lowercase for node paths

# --- Frequency sweep ----------------------------------------------------------
FREQ_START    = 1e3              # Sweep start frequency (Hz)
FREQ_STOP     = 10e6             # Sweep stop  frequency (Hz)
#   Keep FREQ_STOP < 50 MHz / NUM_HARMONICS so all harmonics stay within HF2LI range
NUM_POINTS    = 100              # Number of frequency points
SWEEP_MAPPING = "log"            # "log" (recommended) or "linear"

# --- Signal output ------------------------------------------------------------
OUT_CHANNEL     = 0              # Signal output index (0 or 1)
OUT_MIXER_CH    = 0              # Mixer channel — check LabOne GUI Output tab
DRIVE_AMPLITUDE = 0.5            # Drive amplitude (V peak) — keep within OUT_RANGE
OUT_RANGE       = 1.0            # Output range (V); options: 0.01, 0.1, 1.0, 10.0

# --- Signal input -------------------------------------------------------------
IN_CHANNEL = 0                   # Signal input index (0 or 1)
IN_RANGE   = 1.0                 # Input range (V); set just above expected signal level
IN_AC      = 0                   # 0 = DC coupling, 1 = AC coupling
IN_IMP50   = 0                   # 0 = 1 MΩ input, 1 = 50 Ω input

# --- Demodulators (one per harmonic) -----------------------------------------
OSC_INDEX     = 0                # Oscillator to sweep (0–5 on HF2LI)
NUM_HARMONICS = 4                # Harmonics to measure: 1st, 2nd, 3rd, 4th
DEMOD_TC      = 10e-3            # Filter time constant (s); shorter = faster, noisier
DEMOD_ORDER   = 4                # Filter order (1–8)
DEMOD_RATE    = 100              # Demodulator sample rate (Sa/s)

# --- Sweeper settling and averaging ------------------------------------------
SETTLE_TIME_TC    = 10           # Settle time in multiples of filter TC
SETTLE_INACCURACY = 1e-3         # Settle inaccuracy threshold (0.001 = 0.1 %)
AVG_SAMPLES       = 1            # Number of samples to average per point
AVG_TIME_TC       = 5            # Averaging time in multiples of filter TC
BW_CONTROL        = 1            # 1 = auto-adjust filter BW with frequency (recommended)

# --- Output ------------------------------------------------------------------
SAVE_CSV   = True                # Save results to CSV
SAVE_PLOT  = True                # Save the plot as PNG
OUTPUT_DIR = Path(".")           # Directory for saved files

## 3. Connect to HF2LI

The HF2LI uses its own data server on **port 8005** and only supports **API level 1**.
Use `zhinst.core.ziDAQServer` directly — the high-level toolkit `Session` is not
supported for HF2LI.

In [ ]:
# Port 8005 = HF2 data server, API level 1
daq = zc.ziDAQServer("localhost", 8005, 1)
daq.connectDevice(DEVICE_ID, "USB")

print(f"Connected to {DEVICE_ID.upper()}")
print(f"Sweep: {FREQ_START:.0f} – {FREQ_STOP:.0f} Hz  |  {NUM_POINTS} points  |  {SWEEP_MAPPING}")
print(f"Harmonics: 1 – {NUM_HARMONICS}")

## 4. Configure Signal Output and Input

Nodes are set using `daq.set()` with a list of `(path, value)` tuples — this sends
all settings to the device in a single transaction.

> **Note on `OUT_MIXER_CH`:** In the HF2LI, mixer channel 0 drives the output with
> oscillator 0. If no signal appears, open LabOne GUI → Output tab and check which
> mixer channel is active for your oscillator.

In [ ]:
daq.set([
    [f"/{DEVICE_ID}/oscs/{OSC_INDEX}/freq",                   FREQ_START],
    [f"/{DEVICE_ID}/sigouts/{OUT_CHANNEL}/range",             OUT_RANGE],
    [f"/{DEVICE_ID}/sigouts/{OUT_CHANNEL}/amplitudes/{OUT_MIXER_CH}", DRIVE_AMPLITUDE],
    [f"/{DEVICE_ID}/sigouts/{OUT_CHANNEL}/enables/{OUT_MIXER_CH}",    1],
    [f"/{DEVICE_ID}/sigouts/{OUT_CHANNEL}/on",                1],
    [f"/{DEVICE_ID}/sigins/{IN_CHANNEL}/range",               IN_RANGE],
    [f"/{DEVICE_ID}/sigins/{IN_CHANNEL}/ac",                  IN_AC],
    [f"/{DEVICE_ID}/sigins/{IN_CHANNEL}/imp50",               IN_IMP50],
])

print(f"Output ON  |  amplitude = {DRIVE_AMPLITUDE} V  |  range = ±{OUT_RANGE} V")

## 5. Configure Demodulators

All four demodulators are locked to the **same oscillator** (`OSC_INDEX`).
The `harmonic` node tells each demod to measure at a multiple of the oscillator frequency:

| Demod | Harmonic | Measures at |
|-------|----------|-------------|
| 0 | 1 | 1 × f_drive  (fundamental) |
| 1 | 2 | 2 × f_drive |
| 2 | 3 | 3 × f_drive |
| 3 | 4 | 4 × f_drive |

When the sweeper steps the oscillator frequency, all four reference frequencies
update automatically.

In [ ]:
for i in range(NUM_HARMONICS):
    daq.set([
        [f"/{DEVICE_ID}/demods/{i}/enable",       1],
        [f"/{DEVICE_ID}/demods/{i}/oscselect",    OSC_INDEX],
        [f"/{DEVICE_ID}/demods/{i}/harmonic",     i + 1],
        [f"/{DEVICE_ID}/demods/{i}/adcselect",    IN_CHANNEL],
        [f"/{DEVICE_ID}/demods/{i}/order",        DEMOD_ORDER],
        [f"/{DEVICE_ID}/demods/{i}/timeconstant", DEMOD_TC],
        [f"/{DEVICE_ID}/demods/{i}/rate",         DEMOD_RATE],
        [f"/{DEVICE_ID}/demods/{i}/phaseshift",   0.0],
    ])

suffixes = ["st", "nd", "rd", "th"]
for i in range(NUM_HARMONICS):
    print(f"  Demod {i}  →  {i+1}{suffixes[i]} harmonic  ({i+1} × f_drive)")

## 6. Configure Sweeper

The sweeper is created from the DAQ server with `daq.sweep()` and configured
using `sweeper.set("parameter", value)`.

- **Settling:** waits at each frequency until the filter output is stable
- **Auto bandwidth (`bandwidthcontrol=1`):** scales the filter TC with frequency —
  essential for wide logarithmic sweeps to keep constant frequency resolution

In [ ]:
sweeper = daq.sweep()

sweeper.set("device",                DEVICE_ID)
sweeper.set("gridnode",              f"/{DEVICE_ID}/oscs/{OSC_INDEX}/freq")
sweeper.set("start",                 FREQ_START)
sweeper.set("stop",                  FREQ_STOP)
sweeper.set("samplecount",           NUM_POINTS)
sweeper.set("xmapping",              1 if SWEEP_MAPPING == "log" else 0)  # 0=linear, 1=log

sweeper.set("settling/tc",           SETTLE_TIME_TC)
sweeper.set("settling/inaccuracy",   SETTLE_INACCURACY)
sweeper.set("averaging/tc",          AVG_TIME_TC)
sweeper.set("averaging/sample",      AVG_SAMPLES)
sweeper.set("bandwidthcontrol",      BW_CONTROL)

# Subscribe all four demodulator sample nodes
for i in range(NUM_HARMONICS):
    sweeper.subscribe(f"/{DEVICE_ID}/demods/{i}/sample")

print("Sweeper configured and subscriptions set.")

## 7. Run Sweep

In [ ]:
print(f"Starting sweep  ({NUM_POINTS} points, TC = {DEMOD_TC*1e3:.1f} ms, settle = {SETTLE_TIME_TC} × TC)")
sweeper.execute()
t0 = time.time()

# sweeper.finished() is not available in newer zhinst.core (26.x) where
# daq.sweep() returns a toolkit-wrapped module — use wait_done() instead
sweeper.wait_done(timeout=3600)

print(f"Sweep complete in {time.time()-t0:.1f} s")

data = sweeper.read()       # no argument in toolkit-wrapped sweeper
sweeper.finish()
sweeper.unsubscribe("*")

## 8. Extract Results

`sweeper.read(True)` returns a dict keyed by node path.
`data[node][0][0]` gives the result dict with:
- `"grid"` — frequency axis (Hz)
- `"r"` — amplitude R = √(X² + Y²) in V RMS
- `"phase"` — phase (rad)
- `"x"`, `"y"` — real and imaginary components

In [ ]:
freq_axis  = None
amplitudes = {}     # {harmonic_number: np.array}

for i in range(NUM_HARMONICS):
    node_key      = f"/{DEVICE_ID}/demods/{i}/sample"
    result        = data[node_key][0][0]
    freq_axis     = result["grid"]
    amplitudes[i + 1] = result["r"]

print(f"{len(freq_axis)} points  |  {freq_axis[0]:.1f} – {freq_axis[-1]:.1f} Hz")
for n in range(1, NUM_HARMONICS + 1):
    print(f"  Harmonic {n}: mean amplitude = {np.mean(amplitudes[n])*1e3:.3f} mV")

## 9. Calculate THD

$$\mathrm{THD} = \frac{\sqrt{V_2^2 + V_3^2 + V_4^2}}{V_1} \times 100\,\%$$

where $V_1$ is the fundamental amplitude and $V_2$–$V_4$ are the harmonic amplitudes.

In [ ]:
V1  = amplitudes[1]
THD = 100.0 * np.sqrt(sum(amplitudes[n]**2 for n in range(2, NUM_HARMONICS + 1))) / V1

print(f"THD summary:")
print(f"  Mean : {np.mean(THD):.3f} %")
print(f"  Min  : {np.min(THD):.3f} %  at {freq_axis[np.argmin(THD)]:.1f} Hz")
print(f"  Max  : {np.max(THD):.3f} %  at {freq_axis[np.argmax(THD)]:.1f} Hz")

## 10. Plot Results

In [ ]:
COLORS   = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
SUFFIXES = ["st", "nd", "rd", "th"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
fig.suptitle(f"HF2LI THD Measurement — {DEVICE_ID.upper()}", fontsize=13)

for n in range(1, NUM_HARMONICS + 1):
    ax1.plot(
        freq_axis,
        20 * np.log10(amplitudes[n] + 1e-12),
        color=COLORS[n - 1],
        label=f"{n}{SUFFIXES[n-1]} harmonic  ({n}×f)"
    )
ax1.set_ylabel("Amplitude (dBV)")
ax1.legend()
ax1.grid(True, which="both", alpha=0.4)

ax2.plot(freq_axis, THD, color="black", linewidth=1.5)
ax2.set_ylabel("THD (%)")
ax2.set_xlabel("Drive Frequency (Hz)")
ax2.grid(True, which="both", alpha=0.4)

if SWEEP_MAPPING == "log":
    ax1.set_xscale("log")
    ax2.set_xscale("log")

plt.tight_layout()

if SAVE_PLOT:
    plot_path = OUTPUT_DIR / f"THD_{DEVICE_ID.upper()}.png"
    plt.savefig(plot_path, dpi=150)
    print(f"Plot saved → {plot_path}")

plt.show()

## 11. Save Data to CSV

In [ ]:
if SAVE_CSV:
    csv_path = OUTPUT_DIR / f"THD_{DEVICE_ID.upper()}.csv"
    header = "frequency_Hz," + ",".join(f"harmonic_{n}_V" for n in range(1, NUM_HARMONICS + 1)) + ",THD_percent"
    rows   = np.column_stack(
        [freq_axis] + [amplitudes[n] for n in range(1, NUM_HARMONICS + 1)] + [THD]
    )
    np.savetxt(csv_path, rows, delimiter=",", header=header, comments="")
    print(f"Data saved → {csv_path}")

## 12. Cleanup

In [ ]:
daq.set([[f"/{DEVICE_ID}/sigouts/{OUT_CHANNEL}/on", 0]])
print("Output off. Done.")